# MedicalGPT - Full RLHF Pipeline (SFT → RM → PPO)

Complete training pipeline using **Qwen2.5-3B** with LoRA on A100 GPU.

**Pipeline:**
1. Data Preparation (download & convert datasets)
2. Supervised Fine-Tuning (SFT)
3. Reward Model Training (RM)
4. Model Merging (LoRA → Full)
5. PPO Training (RLHF)
6. Evaluation

**Features:**
- Automatic checkpoint resume (local + HuggingFace Hub)
- wandb logging
- HuggingFace Hub checkpoint upload
- Hyperparameter-based model naming

## 0. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repository
!git clone --depth 1 https://github.com/lldois/medicalgpt.git
%cd medicalgpt
%ls

In [ ]:
# Install dependencies
!pip install -r requirements.txt
!pip install bitsandbytes

In [ ]:
# ============ CONFIGURE YOUR KEYS HERE ============
import os

# HuggingFace Hub token (for uploading checkpoints)
os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # TODO: Replace with your token

# HuggingFace username (for repo naming)
os.environ["HF_USERNAME"] = "YOUR_HF_USERNAME"  # TODO: Replace with your username

# Weights & Biases API key (for training logging)
os.environ["WANDB_API_KEY"] = "YOUR_WANDB_KEY"  # TODO: Replace with your key

# Login to HuggingFace
!huggingface-cli login --token $HF_TOKEN

# Login to wandb
!wandb login $WANDB_API_KEY

print("Environment configured!")

## 1. Prepare Data

Downloads datasets from HuggingFace:
- **SFT**: UltraChat 200k (10K samples)
- **Reward**: Anthropic HH-RLHF (5K samples)
- **Eval**: UltraChat test split (200 samples)

In [ ]:
!python src/prepare_data.py --mode all --data_dir ./data \
    --sft_samples 10000 \
    --reward_samples 5000 \
    --eval_samples 200

In [ ]:
# Verify data
!echo "SFT data:" && wc -l data/finetune/*.jsonl
!echo "Reward data:" && wc -l data/reward/*.jsonl
!echo "Eval data:" && wc -l data/eval/*.jsonl

## 2. Stage 1: Supervised Fine-Tuning (SFT)

LoRA fine-tuning Qwen2.5-3B on instruction-following data.

Expected time: **~3-4 hours** on A100 40GB

In [ ]:
!bash scripts/run_sft.sh

## 3. Stage 2: Reward Model Training

Train reward model on human preference data (pairwise ranking loss).

Expected time: **~2 hours** on A100 40GB

In [ ]:
!bash scripts/run_rm.sh

## 4. Merge LoRA Adapters

Merge LoRA weights into base model for PPO training.

In [ ]:
!bash scripts/run_merge.sh

## 5. Stage 3: PPO Training (RLHF)

Align SFT model using reward model via Proximal Policy Optimization.

Expected time: **~4-5 hours** on A100 40GB

In [ ]:
!bash scripts/run_ppo.sh

## 6. Evaluation

Compute perplexity and generate sample responses.

In [ ]:
# Evaluate SFT model
!bash scripts/run_eval.sh ./merged-sft

In [ ]:
# Evaluate PPO model (if available)
import glob
ppo_dirs = glob.glob("outputs-ppo-*")
if ppo_dirs:
    ppo_dir = sorted(ppo_dirs)[-1]
    print(f"Evaluating PPO model: {ppo_dir}")
    !bash scripts/run_eval.sh {ppo_dir}
else:
    print("No PPO output found")

In [ ]:
# View evaluation results
import json

with open("eval_results/eval_results.json", "r") as f:
    results = json.load(f)

if "perplexity" in results:
    print(f"Perplexity: {results['perplexity']:.4f}")

print("\n=== Generated Samples ===")
for sample in results.get("generated_samples", []):
    print(f"\n[Prompt] {sample['prompt']}")
    print(f"[Response] {sample['response'][:300]}...")
    print("-" * 60)

## 7. (Optional) Run Full Pipeline

You can also run the entire pipeline with a single command:

In [ ]:
# Uncomment to run the full pipeline
# !bash scripts/run_all.sh

## 8. Interactive Demo

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the final model
model_path = "./merged-sft"  # or PPO output
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)

def chat(prompt, max_new_tokens=256):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

# Test
print(chat("What are the symptoms of diabetes?"))